In [1]:
import os
from openai import OpenAI

# 从环境变量获取 DeepSeek API Key
api_key = os.getenv("DEEPSEEK_API_KEY")
if not api_key:
    raise ValueError("请设置 DEEPSEEK_API_KEY 环境变量")

# 初始化 DeepSeek 客户端
client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",  # DeepSeek API 的基地址
)

In [2]:
def send_messages(messages):
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        tools=tools
    )
    return response.choices[0].message

In [3]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "根据地点查询当地的天气，在调用工具时，要求用户必须输入location字段",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    }
                },
                "required": ["location"]
            },
        }
    },
]

In [4]:
messages = [{"role": "user", "content": "北京今天的天气怎么样?"}]
message = send_messages(messages)

In [5]:
print(f"User>\t {messages[0]['content']}, message: {message}")

User>	 北京今天的天气怎么样?, message: ChatCompletionMessage(content='我来帮您查询北京今天的天气情况。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_00_aCjwCHmx9V30qxUfwC2V1NCT', function=Function(arguments='{"location": "北京"}', name='get_weather'), type='function', index=0)])


In [6]:
tool = message.tool_calls[0]
messages.append(message)

In [7]:
tool

ChatCompletionMessageToolCall(id='call_00_aCjwCHmx9V30qxUfwC2V1NCT', function=Function(arguments='{"location": "北京"}', name='get_weather'), type='function', index=0)

In [8]:
# 模拟 search_web 工具调用结果（直接返回24度）
messages.append({"role": "tool", "tool_call_id": tool.id, "content": "24℃"})
print(f"messages content: {messages}\n")
message = send_messages(messages)
print(f"Model>\t {message.content}")

messages content: [{'role': 'user', 'content': '北京今天的天气怎么样?'}, ChatCompletionMessage(content='我来帮您查询北京今天的天气情况。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_00_aCjwCHmx9V30qxUfwC2V1NCT', function=Function(arguments='{"location": "北京"}', name='get_weather'), type='function', index=0)]), {'role': 'tool', 'tool_call_id': 'call_00_aCjwCHmx9V30qxUfwC2V1NCT', 'content': '24℃'}]

Model>	 根据查询结果，北京今天的天气温度是24℃。这是一个比较舒适的温度，适合外出活动。
